# A2 — Delegating Autonomy

**The paradigm shift: from static DAG to dynamic manager-worker delegation.**

A2 introduces the **delegation loop engine** — a fundamentally different execution model:

| | A0/A1 (DAG) | A2 (Delegation Loop) |
|---|---|---|
| **Who decides what runs?** | The YAML graph | A manager agent (LLM) |
| **Agents known upfront?** | Yes, all listed in graph | No — manager spawns workers dynamically |
| **Execution pattern** | Topological sort | Iterative loop: plan → delegate → validate → repeat |
| **Budget controls** | Optional timeouts | **Mandatory** hard limits |
| **Predictability** | Deterministic | Non-deterministic (LLM decides) |

### The Delegation Loop

```
              ┌──────────────────────────────────────┐
              │           MANAGER AGENT               │
              │  (LLM with task + accumulated state)  │
              └──────────┬───────────────┬────────────┘
                         │               │
              ┌──────────▼──┐   ┌────────▼────────┐
              │  Worker A   │   │    Worker B      │
              │  (spawned)  │   │    (spawned)     │
              └──────────┬──┘   └────────┬────────┘
                         │               │
              ┌──────────▼───────────────▼────────┐
              │     MANAGER validates results       │
              │     Budget check → continue/stop    │
              └────────────────────────────────────┘
```

### When to use A2
- Tasks where the number of steps is unknown at design time
- Creative, exploratory, or analytical work
- When you want the LLM to decide what sub-tasks to create
- Any task that benefits from a "manager" decomposing work

## 1. Provider Setup

In [ ]:
PROVIDER = "openrouter"
OLLAMA_MODEL = "qwen3:1.7b"
OLLAMA_BASE_URL = "http://localhost:11434/v1"
OPENROUTER_API_KEY = ""
OPENROUTER_MODEL = "openai/gpt-5-mini"
CUSTOM_API_KEY = ""
CUSTOM_BASE_URL = ""
CUSTOM_MODEL = ""

import os

if PROVIDER == "ollama":
    os.environ["LLM_API_KEY"] = "ollama"
    os.environ["LLM_BASE_URL"] = OLLAMA_BASE_URL
    MODEL = f"ollama/{OLLAMA_MODEL}"
elif PROVIDER == "openrouter":
    key = OPENROUTER_API_KEY or os.getenv("OPENROUTER_API_KEY", "")
    if not key:
        raise ValueError("Set OPENROUTER_API_KEY above or as an environment variable")
    os.environ["LLM_API_KEY"] = key
    os.environ["LLM_BASE_URL"] = "https://openrouter.ai/api/v1"
    MODEL = OPENROUTER_MODEL
elif PROVIDER == "custom":
    key = CUSTOM_API_KEY or os.getenv("LLM_API_KEY", "")
    url = CUSTOM_BASE_URL or os.getenv("LLM_BASE_URL", "")
    if not key or not url:
        raise ValueError("Set CUSTOM_API_KEY and CUSTOM_BASE_URL")
    os.environ["LLM_API_KEY"] = key
    os.environ["LLM_BASE_URL"] = url
    MODEL = CUSTOM_MODEL
else:
    raise ValueError(f"Unknown PROVIDER '{PROVIDER}'")

os.environ["LLM_MODEL"] = MODEL
print(f"Provider: {PROVIDER}  |  Model: {MODEL}")

## 2. The A2 Workflow — YAML Anatomy

The critical change: `engine: delegation_loop` replaces the static `graph`.

```yaml
orchestration:
  engine: delegation_loop          # NEW: dynamic engine

  delegation_loop:
    manager: agents/manager        # LLM-driven manager agent

    budget:                        # MANDATORY at A2: hard limits
      max_loops: 5                 # Max manager iterations
      max_total_workers: 10        # Max workers spawned
      max_total_tokens: 500000     # Token budget
      max_wall_time: 300           # Wall clock seconds
      max_tool_calls: 50           # Total tool invocations
      max_depth: 3                 # Recursion depth (for A4)

    worker_policy:                 # Safety envelope for workers
      enforced:
        sandbox:
          type: subprocess
        forbidden_tools:
          - "file.write_outside_workspace"
          - "shell.execute"

    termination:                   # Stall detection
      enabled: true
      window: 3                    # Check last N iterations
      min_confidence_delta: 0.05   # Min progress per window
      action: warn_then_stop

    validation:                    # Two-tier validation
      deterministic:
        always: true               # Schema checks always run
      llm:
        enabled: true              # LLM validates worker output
        skip_when_confidence_above: 0.95
```

**The budget is not optional.** Without hard limits, an A2 workflow could loop forever. This is AWP's core safety principle: **autonomy requires guardrails.**

## 3. Run an A2 Delegation Loop with AgentWorkflow

The `AgentWorkflow` API is the programmatic way to run delegation loops.
It handles all the YAML setup internally — you just provide the task and constraints.

In [ ]:
import time
from awp.data import AgentWorkflow

TASK = (
    "Analyze the pros and cons of three programming paradigms: "
    "object-oriented, functional, and procedural programming. "
    "For each paradigm, provide: (1) core principles, (2) best use cases, "
    "(3) limitations. Then write a comparison summary. "
    "Save the analysis as 'paradigm_comparison.md' in the output directory."
)

print(f"Task: {TASK[:80]}...")
print(f"Model: {MODEL}")
print(f"Budget: 10 loops, 500K tokens, 120s wall time")
print()

t0 = time.time()

result = AgentWorkflow(
    inputs={"focus": "practical, real-world comparisons"},
    task=TASK,
    model=MODEL,

    # A2 BUDGET — the defining feature
    max_loops=10,
    max_total_tokens=500_000,
    max_wall_time=120,
    max_tool_calls=30,
    max_total_workers=5,
    max_depth=1,              # No recursive delegation (that's A4)

    # Worker capabilities
    code_mode=True,
    tool_creation=False,      # No dynamic tools (that's A3)
    sandbox="subprocess",
    verbose=True,
).run()

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s — Status: {result['status']}")

## 4. Inspect the Results

In [ ]:
import json

meta = result["metadata"]

print("A2 Delegation Loop Results")
print("=" * 60)
print(f"  Status:       {result['status']}")
print(f"  Loops:        {meta['loops']}")
print(f"  Workers:      {meta['workers_spawned']}")
print(f"  Tool calls:   {meta['tool_calls']}")
print(f"  Tokens:       {meta['tokens_used']:,}")
print(f"  Wall time:    {meta['wall_time']:.1f}s")
print(f"  Output dir:   {meta.get('output_dir', 'N/A')}")
print()

# Show the manager's final result
r = result["result"]
if isinstance(r, dict):
    print(f"  Confidence:   {r.get('confidence', 'N/A')}")
    if "summary" in r:
        summary = str(r["summary"])
        if len(summary) > 500:
            summary = summary[:500] + "..."
        print(f"\n  Summary:\n{summary}")

## 5. Budget Enforcement — Why It Matters

The budget is the **safety envelope** for autonomous agents. Without it, a delegation loop
could spawn workers indefinitely, consume unlimited tokens, or run forever.

AWP enforces budgets **unconditionally** — the manager cannot override them.

In [ ]:
# Show budget utilization
meta = result["metadata"]

budgets = [
    ("Loops",      meta["loops"],          10,       "max_loops"),
    ("Workers",    meta["workers_spawned"], 5,        "max_total_workers"),
    ("Tokens",     meta["tokens_used"],     500_000,  "max_total_tokens"),
    ("Tool calls", meta["tool_calls"],      30,       "max_tool_calls"),
    ("Wall time",  meta["wall_time"],       120,      "max_wall_time"),
]

print("Budget Utilization")
print("=" * 60)
print(f"{'Dimension':<15} {'Used':>10} {'Limit':>10} {'Usage':>8}")
print("-" * 60)

for name, used, limit, param in budgets:
    if isinstance(used, float):
        used_str = f"{used:.1f}"
    else:
        used_str = f"{used:,}"
    limit_str = f"{limit:,}"
    pct = (used / limit * 100) if limit > 0 else 0
    bar = "#" * int(pct / 5) + "." * (20 - int(pct / 5))
    print(f"  {name:<13} {used_str:>10} {limit_str:>10} {pct:>6.1f}% [{bar}]")

print()
print("If any dimension hits 100%, the loop terminates with status 'budget_exceeded'.")
print("This is by design — runaway prevention is a core safety guarantee.")

## 6. The Manager's Decision-Making

Unlike A0/A1 where the graph determines execution, at A2 the **manager decides**:
- What workers to spawn
- What instructions each worker receives
- Whether the task is complete or needs more iterations
- Whether to adjust strategy if workers fail

The manager has access to **intelligence features**:

| Feature | What it does |
|---------|-------------|
| **Planning** | Decompose the task into subtasks before delegating |
| **Diagnosis** | Generate hypotheses when workers fail |
| **Strategy switching** | Try different approaches (decompose finer, simplify, reframe) |
| **Decision journal** | Log reasoning for each delegation decision |
| **Budget reservation** | Reserve budget for critical remaining subtasks |

In [ ]:
# Show output artifacts
from pathlib import Path

output_dir = Path(meta.get("output_dir", "/tmp/awp-none"))
output_path = output_dir / "output"

if output_path.exists():
    files = sorted(output_path.rglob("*"))
    print(f"Output artifacts ({len([f for f in files if f.is_file()])} files):")
    for f in files:
        if f.is_file():
            rel = f.relative_to(output_dir)
            sz = f.stat().st_size
            print(f"  {rel} ({sz:,} bytes)")
            if f.suffix == ".md" and sz < 2000:
                print(f"\n--- {f.name} ---")
                print(f.read_text()[:1000])
                print("---\n")
else:
    print("No output directory found (workflow may not have completed).")

## 7. Key Differences: A1 → A2

| Aspect | A1 (Adaptive) | A2 (Delegating) |
|--------|--------------|------------------|
| **Engine** | `dag` (topological sort) | `delegation_loop` (iterative) |
| **Agent creation** | Fixed in YAML | Manager spawns at runtime |
| **Execution path** | Predetermined | LLM-decided per iteration |
| **Budget** | Optional timeouts | **Mandatory** multi-dimensional limits |
| **Validation** | Schema only | Two-tier (deterministic + LLM) |
| **Stall detection** | None | Confidence delta monitoring |
| **Worker policy** | None | Enforced sandbox + forbidden tools |

**The fundamental shift:** A0/A1 workflows are **programs** (the human writes the plan). A2+ workflows are **agents** (the LLM writes the plan at runtime, within budget constraints).

**Next:** Open `A3_self_tooling.ipynb` to see how agents can create their own tools at runtime.